1. 导入库和配置

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from method import (
    TrainDataset, TestDataset, downsample, load_dataset_info,
    get_class_weights, EEGNet
)

# ==================== 配置参数 ====================
DATA_NAME = "SLEEP"
DATA_DIR = Path(DATA_NAME)

# 超参数
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3
TARGET_LEN = 1000  # 降采样后的时间点数量
info = load_dataset_info(DATA_DIR)

2：加载数据集

In [ ]:
# ==================== 加载数据 ====================
train_ds = TrainDataset(DATA_DIR / "train.h5")
val_ds = TrainDataset(DATA_DIR / "val.h5")
test_ds = TestDataset(DATA_DIR / "test_x_only.h5")

print(f"原始形状: {train_ds.x.shape}")
train_ds.x = downsample(train_ds.x, TARGET_LEN)
val_ds.x = downsample(val_ds.x, TARGET_LEN)
print(f"降采样后: {train_ds.x.shape}\n")

# 类别权重
class_weights = get_class_weights(train_ds)
print(f"类别权重: {class_weights.tolist()}\n")

# DataLoader
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

3：创建模型

In [ ]:
# ==================== 创建模型 ====================
CHANNELS = train_ds.x.shape[1]
CLASSES = len(info['category_list'])

model = EEGNet(chans=CHANNELS, time_points=TARGET_LEN, num_classes=CLASSES)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}\n")

4.训练

In [ ]:
# ==================== 训练设置 ====================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.3
)
# ==================== 训练循环 ====================
train_losses, val_losses, val_accs = [], [], []
best_acc = 0

print(f"训练 {EPOCHS} epochs...\n")

for epoch in range(EPOCHS):
    # 训练阶段
    model.train()
    train_loss, train_correct, train_n = 0, 0, 0
    for x, y in train_loader:
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item() * len(y)
        train_n += len(y)
        train_correct += (out.argmax(1) == y).sum().item()

    # 验证阶段
    model.eval()
    val_loss, val_correct, val_n = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * len(y)
            val_n += len(y)
            val_correct += (out.argmax(1) == y).sum().item()

    train_losses.append(train_loss / train_n)
    val_losses.append(val_loss / val_n)
    val_acc = val_correct / val_n
    val_accs.append(val_acc)

    # 保存最佳模型
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), DATA_DIR / 'best_model.pth')

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_losses[-1]:.4f} | "
              f"Val Loss: {val_losses[-1]:.4f} | "
              f"Val Acc: {val_acc:.4f}")

print(f"\n最佳验证准确率: {best_acc:.4f}")
# ==================== 绘制训练曲线 ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.legend(), ax1.grid(True, alpha=0.3)
ax1.set_title('Loss Curve')

ax2.plot(val_accs, color='green', label='Val Accuracy')
ax2.axhline(y=0.2, color='red', linestyle='--', label='Random (20%)')
ax2.legend(), ax2.grid(True, alpha=0.3)
ax2.set_title('Accuracy Curve')

plt.tight_layout()
plt.savefig(DATA_DIR / 'training_curve.png', dpi=150)
plt.show()

Epoch  10/50 | Train Loss: 1.2973 | Val Loss: 1.2485 | Val Acc: 0.5152


5.保存结果

In [ ]:
# -------------------------
# 保存 test 预测标签（每行一个数字）
# -------------------------
model.eval()
output_path = f'{DATA_NAME}/{DATA_NAME}.txt'
all_test_labels = []
with torch.no_grad():
    for test_data in test_loader:  # test_loader 已经是 shuffle=False
        test_output = model(test_data)
        test_pred = torch.argmax(test_output, dim=1)
        all_test_labels.extend(test_pred.cpu().tolist())

with open(output_path, "w", encoding="utf-8") as f:
    for label in all_test_labels:
        f.write(f"{int(label)}\n")

print(f"Saved {len(all_test_labels)} labels to: {output_path}")